# Manual Bakery review

This notebook reviews the bakery verification results from Notebook 08.

The purpose is to identify specific classifications that may require manual checking and to maintain a single manual-review record.

In [1]:
from pathlib import Path
from shutil import copy2

import pandas as pd

# Main verification folder
VERIFICATION_FOLDER = Path("../data/business/interim/ai_verification_v2")

# Notebook 08 output and Notebook 09 output
AI_RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")
MANUAL_REVIEW_FOLDER = (VERIFICATION_FOLDER / "manual_review")
MANUAL_REVIEW_FOLDER.mkdir(parents=True, exist_ok=True)

BACKUP_FOLDER = (MANUAL_REVIEW_FOLDER / "backups")
BACKUP_FOLDER.mkdir(parents=True, exist_ok=True)

MANUAL_REVIEW_PATH = (MANUAL_REVIEW_FOLDER / "bakery_manual_review.xlsx")


EXPECTED_TOTAL = 28847

## Load AI verification results

The AI verification results are loaded as the source dataset for manual-review diagnostics.

In [2]:
ai_results = pd.read_csv(AI_RESULTS_PATH)

ai_results = (ai_results
              .sort_values("BakeryRank")
              .reset_index(drop=True))

print(f"AI verification rows loaded: {len(ai_results)}")

print(f"Highest BakeryRank available: {ai_results['BakeryRank'].max()}")

print(f"Verification completion: {len(ai_results) / EXPECTED_TOTAL:.2%}")

AI verification rows loaded: 28847
Highest BakeryRank available: 28847
Verification completion: 100.00%


## Check loaded verification results

A small number of checks are used to make sure the verification results are suitable for manual review.

In [8]:
ESSENTIAL_COLUMNS = [
    "BusinessName",
    "FHRSIDRep",
    "BusinessType",
    "Address",
    "PostCode",
    "LocalAuthorityName",
    "BakeryRank",
    "BakeryScore",
    "StoreCount",
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus",
    "AIVerdict",
    "AIReason"]

missing_columns = [column
                   for column in ESSENTIAL_COLUMNS
                   if column not in ai_results.columns]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

if ai_results["BakeryRank"].isna().any():
    raise ValueError("Missing BakeryRank values found.")

if ai_results["BakeryRank"].duplicated().any():
    raise ValueError("Duplicate BakeryRank values found.")

if len(ai_results) != EXPECTED_TOTAL:
    raise ValueError(f"Expected {EXPECTED_TOTAL} verification results, found {len(ai_results)}.")

print("Verification results ready for review.")

Verification results ready for review.


## Inspect AI verification results

The overall verification outputs are inspected before any manual-review rules are defined.

This provides an overview of the AI classifications and the individual verification fields used to produce them.

In [4]:
VERIFICATION_COLUMNS = [
    "AIVerdict",
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus"]


for column in VERIFICATION_COLUMNS:

    summary = (ai_results[column]
               .value_counts(dropna=False)
               .rename("Count")
               .to_frame())

    summary["Percent"] = (summary["Count"] / len(ai_results) * 100).round(1)

    print(f"\n{column}")

    display(summary)


AIVerdict


,Count,Percent
AIVerdict,,
NOT_BAKERY,19849,68.8
UNCLEAR,7048,24.4
BAKERY,1950,6.8



LocationMatch


,Count,Percent
LocationMatch,,
YES,22406,77.7
UNCLEAR,6441,22.3



AIStatus


,Count,Percent
AIStatus,,
ACTIVE,21563,74.7
UNCLEAR,6470,22.4
INACTIVE,814,2.8



PhysicalRetail


,Count,Percent
PhysicalRetail,,
NO,10824,37.5
YES,10380,36.0
UNCLEAR,7643,26.5



BakeryFocus


,Count,Percent
BakeryFocus,,
NO,17528,60.8
UNCLEAR,9047,31.4
YES,2272,7.9


### Relationship between verdicts and verification fields

The final AI verdict is compared with each of the individual verification fields.

In [5]:
VERDICT_FIELDS = [
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus"]

for column in VERDICT_FIELDS:
    print(f"\nAIVerdict by {column}")

    comparison = pd.crosstab(ai_results["AIVerdict"], ai_results[column], margins=True)

    display(comparison)


AIVerdict by LocationMatch


LocationMatch,UNCLEAR,YES,All
AIVerdict,,,
BAKERY,0,1950,1950
NOT_BAKERY,0,19849,19849
UNCLEAR,6441,607,7048
All,6441,22406,28847



AIVerdict by AIStatus


AIStatus,ACTIVE,INACTIVE,UNCLEAR,All
AIVerdict,,,,
BAKERY,1950,0,0,1950
NOT_BAKERY,19011,814,24,19849
UNCLEAR,602,0,6446,7048
All,21563,814,6470,28847



AIVerdict by PhysicalRetail


PhysicalRetail,NO,UNCLEAR,YES,All
AIVerdict,,,,
BAKERY,0,0,1950,1950
NOT_BAKERY,10793,822,8234,19849
UNCLEAR,31,6821,196,7048
All,10824,7643,10380,28847



AIVerdict by BakeryFocus


BakeryFocus,NO,UNCLEAR,YES,All
AIVerdict,,,,
BAKERY,0,0,1950,1950
NOT_BAKERY,17528,2044,277,19849
UNCLEAR,0,7003,45,7048
All,17528,9047,2272,28847


### Business types by AI verdict

The FHRS business types are compared across the AI verdicts to understand the
types of establishments appearing in each classification.

In [6]:
business_type_summary = pd.crosstab(ai_results["BusinessType"], ai_results["AIVerdict"], margins=True)

business_type_summary = (business_type_summary
                         .sort_values("BAKERY", ascending=False))

display(business_type_summary)

AIVerdict,BAKERY,NOT_BAKERY,UNCLEAR,All
BusinessType,,,,
All,1950,19849,7048,28847
Restaurant/Cafe/Canteen,927,5577,1074,7578
Retailers - other,504,3739,1617,5860
Takeaway/sandwich shop,266,1192,422,1880
Other catering premises,110,2255,2401,4766
Manufacturers/packers,94,420,188,702
Mobile caterer,32,429,640,1101
Hotel/bed & breakfast/guest house,5,532,28,565
Retailers - supermarkets/hypermarkets,5,242,24,271


### Inspect example classifications

Examples from each AI verdict are inspected to understand how the verification fields and evidence reasons are being used in individual cases.

In [7]:
SAMPLE_COLUMNS = [
    "BakeryRank",
    "BakeryScore",
    "BusinessName",
    "BusinessType",
    "Address",
    "PostCode",
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus",
    "AIVerdict",
    "AIReason"]

for verdict in ["BAKERY", "NOT_BAKERY", "UNCLEAR"]:
    sample = (ai_results[ai_results["AIVerdict"] == verdict]
        .sample(n=10, random_state=42)
        .sort_values("BakeryRank"))

    print(f"\n{verdict}")
    display(sample[SAMPLE_COLUMNS])


BAKERY


,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
1145,1146,0.702132,The crusty loaf,Manufacturers/packers,"158, Southampton Way, London",SE5 7EW,YES,ACTIVE,YES,YES,BAKERY,The Crusty Loaf is a physical bakery selling f...
1160,1161,0.698221,Deja Vu Bakery,Takeaway/sandwich shop,"87 High Street, Walthamstow, London",E17 7DB,YES,ACTIVE,YES,YES,BAKERY,Just Eat menu shows a variety of savoury and s...
1844,1845,0.500999,El gran sazon latino bakery,Takeaway/sandwich shop,"80-82, Walworth Road, London",SE1 6SW,YES,ACTIVE,YES,YES,BAKERY,News articles and menu listings confirm a Colo...
2826,2827,0.257685,The Little Battersea Bakery & Coffee Shop,Restaurant/Cafe/Canteen,"55 Battersea Bridge Road, London, Wandsworth",SW11 3AX,YES,ACTIVE,YES,YES,BAKERY,The official website and reviews confirm a phy...
7177,7178,0.060302,Daily Shot,Restaurant/Cafe/Canteen,"22 Aldensley Road, London",W6 0DH,YES,ACTIVE,YES,YES,BAKERY,Daily Shot is a coffee shop that offers fresh ...
8349,8350,0.050008,Caterplus @ Royal Marsden Hospital Oak (Barist...,Restaurant/Cafe/Canteen,"The Royal Marsden Hospital, Downs Road, Sutton",SM2 5PT,YES,ACTIVE,YES,YES,BAKERY,Caterplus @ Royal Marsden Hospital Oak (Barist...
10174,10175,0.039553,Uzbek Corner Streatham,Restaurant/Cafe/Canteen,"2 Central Parade, Streatham High Road, London",SW16 1HT,YES,ACTIVE,YES,YES,BAKERY,Official website and menu confirm a physical r...
15102,15103,0.026817,Brookes,Restaurant/Cafe/Canteen,"137 High Street, Farnborough, Orpington",BR6 7AZ,YES,ACTIVE,YES,YES,BAKERY,Brookes is a coffee and brunch cafe offering a...
16338,16339,0.024983,Konditor,Takeaway/sandwich shop,"15 Cullum Street, London",EC3M 7JJ,YES,ACTIVE,YES,YES,BAKERY,"Konditor is a physical bakery selling cakes, b..."
19697,19698,0.021214,Gail's St Pancras Station,Restaurant/Cafe/Canteen,"unit 15 St Pancras Station Euston Road, London",N1C 4QP,YES,ACTIVE,YES,YES,BAKERY,Gail's is a well-known bakery chain selling br...



NOT_BAKERY


,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
3678,3679,0.155003,Caterlink @St Joseph's RC Primary,School/college/university,"Caterlink @St Joseph's RC Primary, St Josephs ...",N19 5DT,YES,ACTIVE,NO,NO,NOT_BAKERY,Caterlink @St Joseph's RC Primary is a caterin...
8472,8473,0.049080,Simmons Chemist,Retailers - other,"111 Cockfosters Road, BARNET",EN4 0DA,YES,ACTIVE,NO,NO,NOT_BAKERY,Simmons Chemist is a pharmacy and does not sel...
9929,9930,0.040696,FUSION G WRAPS LTD,Takeaway/sandwich shop,"75 Leather Lane, London",EC1N 7TJ,YES,ACTIVE,NO,UNCLEAR,NOT_BAKERY,Companies House and other sources indicate a v...
17806,17807,0.023069,The Wright Education After School Club,Other catering premises,"Maryland Childrens Centre, Maryland Primary Sc...",E15 1QX,YES,ACTIVE,NO,NO,NOT_BAKERY,The Wright Education After School Club is an a...
18334,18335,0.022570,Tapas Brindisa South Kensington,Restaurant/Cafe/Canteen,"7-9 Exhibition Road, LONDON",SW7 2HE,YES,ACTIVE,YES,NO,NOT_BAKERY,"Tapas Brindisa is a Spanish tapas restaurant, ..."
18352,18353,0.022551,Excel London Hospitality,Other catering premises,"ExCel, 1 Western Gateway, Canning Town, London",E16 1XL,YES,ACTIVE,NO,NO,NOT_BAKERY,Excel London Hospitality operates within the E...
18663,18664,0.022194,Cattleya,Restaurant/Cafe/Canteen,"52 Charlton Church Lane, Charlton, Greenwich",SE7 7AB,YES,ACTIVE,YES,NO,NOT_BAKERY,"Cattleya is a Thai and tapas restaurant, with ..."
19831,19832,0.021077,Mpower,Caring Premises,"22 Bromley Road, London",SE6 2TP,YES,ACTIVE,NO,NO,NOT_BAKERY,"Mpower is a care home, not a retail bakery."
22579,22580,0.018796,Aspen@City Of London Academy,School/college/university,"Aspen@City Of London Academy, City Of London A...",N1 8PQ,YES,ACTIVE,NO,NO,NOT_BAKERY,Aspen@City Of London Academy is a catering ser...
27932,27933,0.015477,GATE HOUSE CHAMBERS,Restaurant/Cafe/Canteen,"1 Lady Hale Gate, London",WC1X 8BS,YES,ACTIVE,NO,NO,NOT_BAKERY,"Gatehouse Chambers is a barristers' chambers, ..."



UNCLEAR


,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
1376,1377,0.637780,The Pink Oyster Home Bakery,Other catering premises,NaN,BR1,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"""The Pink Oyster Home Bakery"" is likely a home..."
1566,1567,0.579424,M&H Cake Company,Other catering premises,NaN,N11,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"The business name is generic, and the partial ..."
4378,4379,0.120729,Delizi by Mirela,Other catering premises,NaN,TW7,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"No reliable information could be found for ""De..."
6128,6129,0.073947,Millet Maduva,Other catering premises,NaN,IG1,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"""Millet Maduva"" is too generic, and the partia..."
6380,6381,0.070327,Park Road Stores LTD,Retailers - supermarkets/hypermarkets,"306 Park Road, Hornsey, London",N8 8LA,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,Multiple businesses with similar names exist a...
10677,10678,0.037613,Made In MDX,Takeaway/sandwich shop,Middlesex University The Burroughs London,NW4 4BT,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"No reliable information found for ""Made In MDX..."
10792,10793,0.037176,Shoreditch Events,Other catering premises,"Railway Arch 108, Cannon Street Road, London",E1 2LY,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"""Shoreditch Events"" is a generic name, and the..."
22374,22375,0.018936,Fettle Fields,School/college/university,NaN,SW20,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"""Fettle Fields"" with only a partial postcode a..."
22645,22646,0.018747,Zadi,Takeaway/sandwich shop,"Unit A 39, Alpha Beta Business Centre, 7 - 11 ...",NW10 6HJ,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"Multiple businesses named ""Zadi"" exist, none o..."
25286,25287,0.016945,Sukrah,Mobile caterer,NaN,RM10,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"""Sukrah"" with postcode RM10 is too generic, an..."


## Investigating potential manual-review groups

At this stage it is important to get a sufficiently manually checked bakery dataset and eliminate any false negatives to get the highest recall dataset while also ensuring we keep false postives to a minimum. As it will be tedious to look through each record of bakery and non-bakery, selecting certain groups that may show the highest chance of false positive and false negatives will be the best way to deal with the review.

## Unclear classiication with matched locations

Oftentimes the UNCLEAR classification is derived from a lack of location or uncertainity about the identity of the store.

In [9]:
matched_unclear = ai_results[(ai_results["AIVerdict"] == "UNCLEAR") & (ai_results["LocationMatch"] == "YES")].copy()

print(f"Matched-location UNCLEAR classifications: {len(matched_unclear)}")

matched_unclear_summary = (matched_unclear
                           .groupby(["AIStatus", "PhysicalRetail", "BakeryFocus"]).size()
                           .reset_index(name="Count")
                           .sort_values("Count", ascending=False))

display(matched_unclear_summary)

Matched-location UNCLEAR classifications: 607


,AIStatus,PhysicalRetail,BakeryFocus,Count
1,ACTIVE,UNCLEAR,UNCLEAR,323
3,ACTIVE,YES,UNCLEAR,196
2,ACTIVE,UNCLEAR,YES,42
0,ACTIVE,NO,UNCLEAR,31
4,UNCLEAR,UNCLEAR,UNCLEAR,15


In [10]:
display(matched_unclear
        .sample(n=min(15, len(matched_unclear)), random_state=42)
        [SAMPLE_COLUMNS].sort_values("BakeryRank"))

,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
562,563,0.864298,Kristies cakes and pastries,Retailers - other,"Unit D, 133, Sumner Road, London",SE15 6JL,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,FHRS and business listings confirm the address...
2822,2823,0.258554,Fola Up LTD,Retailers - other,"5 Chequers Parade, Ripple Road, Dagenham",RM9 6RT,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,"Companies House and FHRS confirm the business,..."
3339,3340,0.188211,Gourmet Gurus Ltd,Restaurant/Cafe/Canteen,"143 Coombe Lane, Raynes Park, Merton",SW20 0QX,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,"Gourmet Gurus Ltd is a restaurant/cafe, but th..."
3453,3454,0.176047,Papers & Booze Ltd,Retailers - other,"82 Nelson Road, Twickenham, Richmond Upon Thames",TW2 7AY,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,"""Papers & Booze Ltd"" at a residential address ..."
3458,3459,0.175614,Chocof,Restaurant/Cafe/Canteen,Temple Fortune Parade Finchley Road London,NW11 0QS,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,"Chocof is a cafe that offers ""surprise bags"" o..."
3793,3794,0.147891,Muffims Ltd,Restaurant/Cafe/Canteen,"99 DUNSMURE ROAD BASEMENT, Hackney, London",N16 5HT,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,"Muffims Ltd is listed as a restaurant/cafe, bu..."
5001,5002,0.098770,Skomoroh Ltd,Retailers - other,"149A Albert Road, North Woolwich, London",E16 2JD,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,Skomoroh Ltd is registered as a retailer of fo...
6934,6935,0.062888,Focaccia's,Restaurant/Cafe/Canteen,"54 Warren Street, London",W1T 5NN,YES,ACTIVE,NO,UNCLEAR,UNCLEAR,Focaccia's at this address appears to be a bub...
8044,8045,0.052243,Shrinath Cash & Carry Ltd,Retailers - other,"Unit 12 New College Parade, Finchley Road, London",NW3 5EP,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,"Shrinath Cash & Carry Ltd is a cash and carry,..."
11410,11411,0.035050,Hammersmith Lab,Restaurant/Cafe/Canteen,"26 - 28 Hammersmith Grove, London",W6 7HA,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,Hammersmith Lab is a cafe within an office bui...


### Matched-location UNCLEAR cases with one other condition

In [11]:
matched_unclear_narrow = matched_unclear[
    (matched_unclear["AIStatus"] == "ACTIVE") 
    & ~((matched_unclear["PhysicalRetail"] == "UNCLEAR") 
       & (matched_unclear["BakeryFocus"] == "UNCLEAR"))].copy()

print(f"Matched-location UNCLEAR with at least one condition established: {len(matched_unclear_narrow)}")

display(matched_unclear_narrow
        .groupby(["PhysicalRetail", "BakeryFocus"]).size()
        .reset_index(name="Count")
        .sort_values("Count", ascending=False))

Matched-location UNCLEAR with at least one condition established: 269


,PhysicalRetail,BakeryFocus,Count
2,YES,UNCLEAR,196
1,UNCLEAR,YES,42
0,NO,UNCLEAR,31


In [12]:
display(matched_unclear_narrow
        .sample(n=min(15, len(matched_unclear_narrow)), random_state=42)
        [SAMPLE_COLUMNS].sort_values("BakeryRank"))

,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
3210,3211,0.201292,Niji UK Limited,Retailers - other,"Harrods, 87-135 Brompton Road, LONDON",SW1X 7XL,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,"Niji UK Limited is located within Harrods, whi..."
4502,4503,0.115950,Niki's Homemade Biscuits,Mobile caterer,NaN,EN1,YES,ACTIVE,UNCLEAR,YES,UNCLEAR,Niki's Homemade Biscuits sells biscuits online...
7410,7411,0.057939,Snack Heath,Retailers - other,"49 Montpelier Vale, Blackheath, London",SE3 0TJ,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,Uber Eats lists Snack Heath as an off-licence ...
7769,7770,0.054741,KENTISH BUBBLE LTD,Restaurant/Cafe/Canteen,"Flat C, 102 Queen's Crescent, London",NW5 4DU,YES,ACTIVE,NO,UNCLEAR,UNCLEAR,KENTISH BUBBLE LTD is registered at a resident...
11567,11568,0.034537,Nono camberwell,Retailers - other,"Unit 2, 24, Brunswick Park, London",SE5 7RH,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,"""Nono Camberwell"" is a coffee shop, but it's u..."
13157,13158,0.030461,A-1 Peanuts,Takeaway/sandwich shop,"South Harrow Market, Northolt Road",HA2 0EU,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,"A-1 Peanuts is located in South Harrow Market,..."
13898,13899,0.028929,Nourished Communities,Retailers - other,"12 Blackhorse Lane, Walthamstow, London",E17 6HJ,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,Nourished Communities is a retailer with physi...
16905,16906,0.024223,Fable,Restaurant/Cafe/Canteen,"9 Station Approach, Kew, Richmond Upon Thames",TW9 3QB,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,"Fable is a cafe and restaurant, but the extent..."
17485,17486,0.023483,Betty's,Restaurant/Cafe/Canteen,"32 Plaistow Lane, Bromley",BR1 3PA,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,"Betty's is a new coffee bar and bottle shop, b..."
18100,18101,0.022770,Progress Carib Cafe LTD,Takeaway/sandwich shop,"248 High Road, Harrow",HA3 7BB,YES,ACTIVE,YES,UNCLEAR,UNCLEAR,Progress Carib Cafe is a takeaway/sandwich sho...


### NOT_BAKERY classifications excluded by physical-retail status

In [14]:
physical_retail_exclusions = ai_results[(ai_results["AIVerdict"] == "NOT_BAKERY") 
                                        & (ai_results["LocationMatch"] == "YES") 
                                        & (ai_results["AIStatus"] == "ACTIVE") 
                                        & (ai_results["PhysicalRetail"] == "NO") 
                                        & (ai_results["BakeryFocus"] == "YES")].copy()

print(f"NOT_BAKERY classifications excluded by physical retail: {len(physical_retail_exclusions)}")

NOT_BAKERY classifications excluded by physical retail: 277


In [15]:
display(physical_retail_exclusions
        .sample(n=min(15, len(physical_retail_exclusions)), random_state=42)
        [SAMPLE_COLUMNS].sort_values("BakeryRank"))

,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
382,383,0.913117,Afgani Bakery Ltd,Manufacturers/packers,"17 Pop In Commercail Centre, South Way, Wembley",HA9 0HB,YES,ACTIVE,NO,YES,NOT_BAKERY,Afghani Bakery Ltd is registered as a manufact...
1831,1832,0.504042,Well Bread,Other catering premises,NaN,TW11,YES,ACTIVE,NO,YES,NOT_BAKERY,"Well Bread operates from a private address, of..."
2302,2303,0.376592,Little House Baker,Other catering premises,NaN,N12,YES,ACTIVE,NO,YES,NOT_BAKERY,Little House Baker is a microbaker operating f...
2392,2393,0.355503,Crystal Patisserie,Manufacturers/packers,"Unit 17 Leyton Business Centre Etloe Road, Ley...",E10 7BT,YES,ACTIVE,NO,YES,NOT_BAKERY,Crystal Patisserie is a manufacturer of pastry...
3612,3613,0.160792,Kentish Cookie Company,Retailers - other,NaN,DA7,YES,ACTIVE,NO,YES,NOT_BAKERY,Kentish Cookie Company operates from a private...
4311,4312,0.123073,Rainforest Creations,Manufacturers/packers,"Unit B28 Alpha Beta Business Centre, 7-11 Mine...",NW10 6HJ,YES,ACTIVE,NO,YES,NOT_BAKERY,Rainforest Creations is a catering company and...
4415,4416,0.119282,London basque kitchen ltd,Other catering premises,"Unit 12 Canterbury Industrial, 297, Ilderton R...",SE15 1NP,YES,ACTIVE,NO,YES,NOT_BAKERY,London Basque Kitchen is a bespoke catering co...
11237,11238,0.035618,Treed Wellness Ltd,Retailers - other,NaN,W9 3,YES,ACTIVE,NO,YES,NOT_BAKERY,Treed Wellness Ltd is registered at a private ...
11731,11732,0.034102,Comfort and joy,Takeaway/sandwich shop,"North Cross Road Market, North Cross Road, London",SE22 9ET,YES,ACTIVE,NO,YES,NOT_BAKERY,Comfort and Joy is a home bakery that operates...
12147,12148,0.032942,BISQUOTES,Caring Premises,NaN,NW5,YES,ACTIVE,NO,YES,NOT_BAKERY,"BisQuotes makes artisanal, customised biscuits..."


### Physical-retail exclusions based on absence of evidence